# CAS BDAI CUP SUBMISSION NOTEBOOK



# Import Data

In [11]:
#import data
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

train = pd.read_csv("https://github.com/casbdai/notebooks/raw/main/Module3/99_CAS_BDAI_CUP/train.csv")
test = pd.read_csv("https://github.com/casbdai/notebooks/raw/main/Module3/99_CAS_BDAI_CUP/test.csv")

In [12]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30857 entries, 0 to 30856
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         30857 non-null  object 
 1   year          30857 non-null  int64  
 2   price         30857 non-null  int64  
 3   transmission  30857 non-null  object 
 4   mileage       30857 non-null  int64  
 5   fuelType      30857 non-null  object 
 6   tax           30857 non-null  int64  
 7   mpg           30857 non-null  float64
 8   engineSize    30857 non-null  float64
 9   brand         30857 non-null  object 
 10  ID            30857 non-null  int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 2.6+ MB


In [13]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15896 entries, 0 to 15895
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         15896 non-null  object 
 1   year          15896 non-null  int64  
 2   transmission  15896 non-null  object 
 3   mileage       15896 non-null  int64  
 4   fuelType      15896 non-null  object 
 5   tax           15896 non-null  int64  
 6   mpg           15896 non-null  float64
 7   engineSize    15896 non-null  float64
 8   brand         15896 non-null  object 
 9   ID            15896 non-null  int64  
dtypes: float64(2), int64(4), object(4)
memory usage: 1.2+ MB


In [14]:
#encode categorical data
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Build Model

In [15]:
# Import Functions
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Instantiate Model
model = LinearRegression()

# Create Train Data
X = train.drop("price", axis=1)
y = train["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12)

# fit model
model.fit(X_train, y_train)

#make prediction
y_pred = model.predict(X_test)

# Evaluate Model Performance
root_mean_squared_error(y_test, y_pred)

4152.394556077297

# Save Results for Submission

Make predictions on the competition data with your trained model

In [16]:
test_predictions= model.predict(test)

In [17]:
file_name = "IvoTestSubmission.csv"

In [18]:
def save_submission_for_kaggle(file_name, test_predictions, test):
  import pandas as pd
  submission_data = pd.DataFrame({"ID": test["ID"], "Actual": test_predictions})
  submission_data.to_csv(file_name, index=False)

## Save submission file

In [19]:
save_submission_for_kaggle(file_name, test_predictions, test)

## FOR GOOGLE COLAB USERS ONLY: Download the created file

In [20]:
try:
  from google.colab import files
  files.download(file_name)
except ModuleNotFoundError:
  print("Not using Google Colab")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## FOR ANACONDA USERS ONLY: Find the created file in your folder structure

The file is located in the same directory as your notebook.

In [21]:
# run this cell if you don't know the location
import os
print(os.getcwd())

/content


# How to get Going

- Try out other algorithms!
- Try out Cross Validation and Hyperparameter Tuning (see coding hint below)
- Try to understand why different models perform better or worse. Make Visualizations (Actual vs. Predicted Plots, Feature Importances, etc.)
- Try to make ensemble different predictions (average of multiple models)

# Implementation Help for Grid Search

In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, make_scorer

# fit model
RMSE = make_scorer(root_mean_squared_error, greater_is_better=False) #we create an RMSE scoring function
parameters = {"max_depth": [5,10]} # hyperparameters to be optimized
model_CV = GridSearchCV(DecisionTreeRegressor(), parameters, scoring=RMSE, cv=5, verbose=3) # Apply 5 Cross Validiation Folds to find best hyperparameters

Useful parameters:
- cv: specify the number of cross validation folds
- scoring: specify what score should be used: either custom scoring function like the RSME above or an already implemented scorer like scoring="accuracy" or "recall", or "precision",
- verbose: see the progress of the operation, e.g., verbose=3

After fitting the grid search cross-validation on the training data, you can use the "best_params_" attribute to display the best hyperparameter combination found in the grid search.

In [23]:
model_CV.fit(X, y)

Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV 1/5] END ...................max_depth=5;, score=-4720.196 total time=   0.1s
[CV 2/5] END ...................max_depth=5;, score=-4300.000 total time=   0.1s
[CV 3/5] END ...................max_depth=5;, score=-4143.366 total time=   0.1s
[CV 4/5] END ...................max_depth=5;, score=-4155.609 total time=   0.1s
[CV 5/5] END ...................max_depth=5;, score=-4199.973 total time=   0.1s
[CV 1/5] END ..................max_depth=10;, score=-3595.036 total time=   0.1s
[CV 2/5] END ..................max_depth=10;, score=-2985.431 total time=   0.1s
[CV 3/5] END ..................max_depth=10;, score=-2863.955 total time=   0.1s
[CV 4/5] END ..................max_depth=10;, score=-2944.137 total time=   0.1s
[CV 5/5] END ..................max_depth=10;, score=-2965.819 total time=   0.1s


GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [5, 10]},
             scoring=make_scorer(root_mean_squared_error, greater_is_better=False, response_method='predict'),
             verbose=3)

Get the best score based on the cross validation. It is the mean of the five splits for the best parameter combination.

It is displayed as negative due to implementation reasons and the greater_is_better=False

In [24]:
model_CV.best_score_

np.float64(-3070.8755954664916)

Get detailed results. Rank 1 will have the lowest mean_test_score

In [25]:
pd.DataFrame(model_CV.cv_results_).sort_values(by="rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
1,0.123337,0.001822,0.002814,0.000134,10,{'max_depth': 10},-3595.035612,-2985.430999,-2863.955114,-2944.137157,-2965.819096,-3070.875595,265.314940,1
0,0.072713,0.003304,0.002464,0.000061,5,{'max_depth': 5},-4720.196275,-4300.000489,-4143.365608,-4155.608539,-4199.973456,-4303.828873,215.350944,2


By relying on the mean of the five splits you generalize beyond the single split done in train_test_split.

# Implementation Help for Plotting Feature Importances in Tree-based Models

In [26]:
def plot_variable_importance(model, X_train):

    import matplotlib.pyplot as plt

    from pandas import DataFrame

    imp=DataFrame({"imp":model.feature_importances_, "names":X_train.columns}).sort_values("imp", ascending=True)

    fig, ax = plt.subplots(figsize=(imp.shape[0]/6,imp.shape[0]/5), dpi=300)

    ax.barh(imp["names"],imp["imp"], color="green")

    ax.set_xlabel('\nVariable Importance')

    ax.set_ylabel('Features\n')

    ax.set_title('Variable Importance Plot\n')

    plt.show()

plot_variable_importance(model, X_train)


AttributeError: 'LinearRegression' object has no attribute 'feature_importances_'